In [37]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from torch.optim import AdamW

In [38]:
train_df = pd.read_csv('nlp_train.csv')
test_df  = pd.read_csv('nlp_test.csv')

print(f'Train: {len(train_df)} | Test: {len(test_df)}')
print(train_df.head(3))

Train: 3684 | Test: 1995
                                                text    risk
0  Dermatology case: 55 year old male patient. Le...    high
1  Dermatology case: 85 year old female patient. ...    high
2  Dermatology case: 65 year old female patient. ...  medium


In [39]:
le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['risk'])
test_df['label']  = le.transform(test_df['risk'])

print('Classes:', le.classes_)

Classes: ['high' 'low' 'medium']


In [40]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(texts, labels, tokenizer, max_length=64):
    encoded = tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )
    return {
        'input_ids':      encoded['input_ids'],
        'attention_mask': encoded['attention_mask'],
        'labels':         torch.tensor(labels, dtype=torch.long)
    }

train_data = tokenize(train_df['text'].tolist(), train_df['label'].tolist(), tokenizer)
test_data  = tokenize(test_df['text'].tolist(),  test_df['label'].tolist(),  tokenizer)

In [41]:
train_dataset = TensorDataset(
    train_data['input_ids'],
    train_data['attention_mask'],
    train_data['labels']
)

test_dataset = TensorDataset(
    test_data['input_ids'],
    test_data['attention_mask'],
    test_data['labels']
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Test batches : {len(test_loader)}')

Train batches: 231
Test batches : 125


In [42]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

classes = np.unique(train_df['label'])
weights = compute_class_weight('balanced', classes=classes, y=train_df['label'])
weights_tensor = torch.tensor(weights, dtype=torch.float).to(device)

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3
)
model = model.to(device)

criterion = torch.nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = AdamW(model.parameters(), lr=3e-5)

Device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [46]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0

    for input_ids, attention_mask, labels in loader:
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels         = labels.to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct    += (outputs.logits.argmax(dim=1) == labels).sum().item()

    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc


EPOCHS    = 30
best_acc  = 0
patience  = 3
no_improve = 0

for epoch in range(EPOCHS):
    loss, acc = train_epoch(model, train_loader, optimizer, criterion)
    print(f'Epoch {epoch+1}/{EPOCHS} — Loss: {loss:.4f} | Acc: {acc:.4f}')

    if acc > best_acc:
        best_acc   = acc
        no_improve = 0
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'Best saved (acc: {best_acc:.4f})')
    else:
        no_improve += 1
        print(f'No improve {no_improve}/{patience}')
        if no_improve >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

model.load_state_dict(torch.load('best_model.pt'))
print(f'Best Acc: {best_acc:.4f}')

Epoch 1/30 — Loss: 0.6685 | Acc: 0.6911
Best saved (acc: 0.6911)
Epoch 2/30 — Loss: 0.6674 | Acc: 0.6884
No improve 1/3
Epoch 3/30 — Loss: 0.6584 | Acc: 0.6922
Best saved (acc: 0.6922)
Epoch 4/30 — Loss: 0.6600 | Acc: 0.6927
Best saved (acc: 0.6927)
Epoch 5/30 — Loss: 0.6633 | Acc: 0.6927
No improve 1/3
Epoch 6/30 — Loss: 0.7422 | Acc: 0.6257
No improve 2/3
Epoch 7/30 — Loss: 0.8403 | Acc: 0.5505
No improve 3/3
Early stopping at epoch 7
Best Acc: 0.6927


In [44]:
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for input_ids, attention_mask, labels in loader:
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = outputs.logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    print(classification_report(all_labels, all_preds, target_names=le.classes_))

evaluate(model, test_loader)

              precision    recall  f1-score   support

        high       0.21      0.43      0.28       218
         low       0.96      0.65      0.78      1583
      medium       0.22      0.54      0.31       194

    accuracy                           0.61      1995
   macro avg       0.46      0.54      0.46      1995
weighted avg       0.81      0.61      0.68      1995



In [47]:
import pickle

model.save_pretrained('nlp_model')
tokenizer.save_pretrained('nlp_model')

with open('nlp_model/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [48]:
!zip -r nlp_model.zip nlp_model

  adding: nlp_model/ (stored 0%)
  adding: nlp_model/label_encoder.pkl (deflated 15%)
  adding: nlp_model/config.json (deflated 54%)
  adding: nlp_model/tokenizer_config.json (deflated 43%)
  adding: nlp_model/tokenizer.json (deflated 71%)
  adding: nlp_model/model.safetensors (deflated 7%)


In [49]:
from google.colab import files

files.download('nlp_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>